# 03 — Cold-Start Demonstration
Shows all 4 cold-start paths using real catalog data — no fake users or ratings
are ever synthesized.

In [ ]:
import os, sys
sys.path.insert(0, '..')
import pandas as pd
from src.data import loader
from src.data.cleaner import clean_pipeline
from src.data.splitter import train_val_test_split
from src.models.popularity import PopularityRecommender
from src.models.content_based import ContentBasedRecommender
from src.coldstart.strategies import ColdStartManager

ratings = clean_pipeline(loader.load_ratings())
movies = loader.load_movies()
tags = loader.load_tags()
train, val, test = train_val_test_split(ratings)

pop = PopularityRecommender().fit(train)
cb = ContentBasedRecommender().fit(movies, tags)
coldstart = ColdStartManager(cb, pop)
print("Models ready.")


## 1) New user — picks real movies they like

In [ ]:
liked_titles = ['Toy Story (1995)', 'Jumanji (1995)']
liked_ids = movies[movies['title'].isin(liked_titles)]['movieId'].tolist()
print(f"Liked movie IDs: {liked_ids}")

recs = coldstart.new_user_from_selections(liked_ids, k=10)
recs.merge(movies, on='movieId')[['clean_title', 'similarity', 'final_score', 'explanation']]


## 2) New user — picks genres instead

In [ ]:
recs = coldstart.new_user_from_genres(['Animation', 'Comedy'], movies, k=10)
recs.merge(movies, on='movieId', suffixes=('','_dup'))[['clean_title', 'weighted_rating', 'explanation']]


## 3) Unknown user (no ID, no preferences at all)

In [ ]:
recs = coldstart.unknown_user(k=5)
recs.merge(movies, on='movieId')[['clean_title', 'weighted_rating', 'explanation']]


## 4) New item — a movie with no rating history yet

In [ ]:
# Simulate a 'new' item by picking one with very few ratings in the training set
rating_counts = train.groupby('movieId').size()
rare_movie_id = rating_counts[rating_counts <= 2].index[0]
rare_title = movies[movies.movieId == rare_movie_id]['clean_title'].iloc[0]
print(f"Treating '{rare_title}' (movieId={rare_movie_id}, {rating_counts[rare_movie_id]} ratings) as a new item.")

recs = coldstart.new_item_similarity(rare_movie_id, k=8)
recs.merge(movies, on='movieId')[['clean_title', 'similarity']]


## Summary
| Scenario | Strategy | No fake data because... |
|---|---|---|
| New user, picked movies | Content profile → kNN → popularity re-rank | Profile built only from movies the user actually selected |
| New user, picked genres | Popularity within genre filter | Uses real historical ratings, filtered |
| Unknown user | Popularity baseline | No assumption about the user at all |
| New item | Pure content similarity | No interactions needed — genre/tag metadata only |
